# Inverse 1-SD Move Calculator + Range Probability

Given ATM IV, futures price, and DTE, this notebook computes:
- implied 1-sigma lower/upper strikes
- probability that expiry settles between any two strikes.

In [ ]:
import math
import pandas as pd


def _norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def inverse_1sd_and_range_prob(
    atm_iv,
    futures_price,
    dte_days,
    strike_1,
    strike_2,
    model="lognormal",
    annual_days=365,
):
    """
    Parameters
    ----------
    atm_iv : float
        ATM implied vol (annualized). Accepts 0.16 or 16 (percent).
    futures_price : float
        Current futures price (F).
    dte_days : int
        Days to expiry.
    strike_1, strike_2 : float
        Two strikes for probability-range calculation.
    model : str
        "lognormal" (Black-style) or "normal" (price-normal approximation).
    annual_days : int
        Day count for annualization (365 or 252).

    Returns
    -------
    dict with 1-sigma strikes and range probability.
    """
    if dte_days <= 0:
        raise ValueError("dte_days must be > 0")
    if futures_price <= 0:
        raise ValueError("futures_price must be > 0")

    iv = float(atm_iv)
    if iv > 1.0:
        iv = iv / 100.0
    if iv <= 0:
        raise ValueError("atm_iv must be > 0")

    T = float(dte_days) / float(annual_days)
    sigma_t = iv * math.sqrt(T)

    k_low = min(float(strike_1), float(strike_2))
    k_high = max(float(strike_1), float(strike_2))

    if model.lower() == "normal":
        # Price-normal approximation
        std_price = futures_price * sigma_t
        lower_1sd = futures_price - std_price
        upper_1sd = futures_price + std_price

        z_low = (k_low - futures_price) / std_price
        z_high = (k_high - futures_price) / std_price
        prob_range = _norm_cdf(z_high) - _norm_cdf(z_low)

    elif model.lower() == "lognormal":
        # Black-style: ln(ST/F) ~ N(-0.5*sigma_t^2, sigma_t^2)
        lower_1sd = futures_price * math.exp(-sigma_t)
        upper_1sd = futures_price * math.exp(+sigma_t)

        if k_low <= 0:
            raise ValueError("Strikes must be > 0 for lognormal model")

        mu = -0.5 * sigma_t * sigma_t
        z_low = (math.log(k_low / futures_price) - mu) / sigma_t
        z_high = (math.log(k_high / futures_price) - mu) / sigma_t
        prob_range = _norm_cdf(z_high) - _norm_cdf(z_low)

    else:
        raise ValueError("model must be 'lognormal' or 'normal'")

    return {
        "model": model.lower(),
        "futures_price": float(futures_price),
        "atm_iv_input": float(atm_iv),
        "atm_iv_decimal": iv,
        "dte_days": int(dte_days),
        "time_to_expiry_years": T,
        "sigma_sqrt_t": sigma_t,
        "one_sd_lower_strike": lower_1sd,
        "one_sd_upper_strike": upper_1sd,
        "range_low_strike": k_low,
        "range_high_strike": k_high,
        "probability_between_strikes": prob_range,
    }


# Example usage
out = inverse_1sd_and_range_prob(
    atm_iv=18,           # or 0.18
    futures_price=2450,
    dte_days=12,
    strike_1=2380,
    strike_2=2520,
    model="lognormal",  # "normal" also supported
)

pd.Series(out)

In [ ]:
# Helper wrapper if you only want key outputs quickly

def quick_calc(atm_iv, futures_price, dte_days, strike_1, strike_2):
    out = inverse_1sd_and_range_prob(
        atm_iv=atm_iv,
        futures_price=futures_price,
        dte_days=dte_days,
        strike_1=strike_1,
        strike_2=strike_2,
        model="lognormal",
    )
    print(f"1SD implied range: {out['one_sd_lower_strike']:.2f} to {out['one_sd_upper_strike']:.2f}")
    print(f"P({out['range_low_strike']:.2f} <= ST <= {out['range_high_strike']:.2f}) = {100*out['probability_between_strikes']:.2f}%")


# quick_calc(18, 2450, 12, 2380, 2520)